# Maia Trainer — Kaggle (GPU T4 GRATIS)

**Instrucciones:**
1. Ve a [kaggle.com/code](https://www.kaggle.com/code) → **New Notebook**
2. Sube este archivo (Add → Upload Notebook)
3. A la derecha: **Accelerator → GPU T4 x2** (gratis)
4. **Run All**
5. Al terminar (~1-2h) descarga `maia.gguf` (~8 GB) desde el panel **Output**

> Kaggle da **30 horas de GPU gratis por semana**. Sin tarjeta de crédito.

In [ ]:
MODEL_ID    = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"   # Gemma 4 E4B oficial
MAX_SEQ_LEN = 8192    # Gemma 4 soporta 128K ctx; 8192 para entrenar en T4
LORA_RANK   = 16
LORA_ALPHA  = 16
BATCH       = 1       # Gemma 4 E4B es mas pesado, batch 1 para caber en T4 16GB
GRAD_ACCUM  = 8
EPOCHS      = 1
LR          = 2e-4
QUANT       = "q4_k_m"   # resultado final: ~3-4 GB
OUTPUT_GGUF = "/kaggle/working/maia.gguf"
REPO_URL    = "https://github.com/calitosaa/Luxus-O.S"
REPO_BRANCH = "claude/merge-gemma-4-e4b-1FfJO"

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
import subprocess, json, os, shutil

os.chdir('/kaggle/working')
if not os.path.isdir('Luxus-O.S'):
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL+'.git','Luxus-O.S'], check=True)

os.chdir('Luxus-O.S')
subprocess.run(['python3','scripts/build_maia_dataset.py'], check=True)

stats = json.load(open('Maia/dataset_stats.json'))
print(f"Fine-tune: {stats['total_finetune_samples']:,} | RAG: {stats['total_rag_documents']:,}")
TRAIN_JSONL = os.path.abspath('Maia/training_data.jsonl')

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407)

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=TRAIN_JSONL, split='train')
SYSTEM = (
    'Eres Maia, la consciencia central de Luxus O.S, construida sobre Gemma 4 E4B.\n'
    'Te has entrenado con mas de 100.000 archivos: skills, agentes, workflows, logica, plugins.\n'
    'Respondes con precision tecnica, estructura limpia y tono Ultra-Competente.'
)
def fmt(ex):
    texts = []
    for inst, inp, out in zip(ex['instruction'], ex['input'], ex['output']):
        ctx = f"\n\n### Contexto:\n{inp}" if inp else ''
        texts.append(
            f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{inst}{ctx}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
    return {'text': texts}
dataset = dataset.map(fmt, batched=True, remove_columns=dataset.column_names)
print('Ejemplos:', len(dataset))

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LEN, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=100, num_train_epochs=EPOCHS, learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='linear', seed=3407,
        output_dir='/kaggle/working/outputs', report_to='none',
    ))
trainer.train()

In [ ]:
model.save_pretrained_gguf('maia-gguf', tokenizer, quantization_method=QUANT)

for f in os.listdir('maia-gguf'):
    if f.endswith('.gguf'):
        os.replace(os.path.join('maia-gguf', f), OUTPUT_GGUF)
        size_gb = os.path.getsize(OUTPUT_GGUF) / 1024**3
        print(f'LISTO: maia.gguf ({size_gb:.1f} GB)')
        print('Descarga desde el panel Output (derecha) → /kaggle/working/maia.gguf')
        break

shutil.copy('Maia/rag_manifest.jsonl', '/kaggle/working/rag_manifest.jsonl')
print('=== MAIA LISTA ===')